In [8]:
import langchain


In [9]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

True

In [10]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### EXample 1: Simple LLM call with streaming

In [11]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [12]:
model=init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019A8FF0A510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019A8FF0B230>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [13]:
## create messages
messages = [
    SystemMessage("You are a helpful AI Assistant"),
    HumanMessage("What are the top 3 benefits of using Langchain?")
]

In [14]:
##invoke the model
response = model.invoke(messages)
response

AIMessage(content="Langchain is a platform that allows users to build, compose, and connect different AI models and services to create more complex and powerful applications. Here are the top 3 benefits of using Langchain:\n\n1. **Composition of AI Models**: Langchain enables users to combine multiple AI models from different sources, including internal models, external APIs, and databases. This allows users to leverage the strengths of different models, creating a more robust and accurate application. By composing AI models, users can create solutions that are tailored to their specific needs and use cases.\n\n2. **Efficient Knowledge Retrieval**: Langchain provides a powerful knowledge retrieval system that enables users to access and combine information from various sources, including text, images, and other data types. This allows users to create applications that can efficiently retrieve and process information, making them more productive and effective.\n\n3. **Customizable and I

In [15]:
print(response.content)

Langchain is a platform that allows users to build, compose, and connect different AI models and services to create more complex and powerful applications. Here are the top 3 benefits of using Langchain:

1. **Composition of AI Models**: Langchain enables users to combine multiple AI models from different sources, including internal models, external APIs, and databases. This allows users to leverage the strengths of different models, creating a more robust and accurate application. By composing AI models, users can create solutions that are tailored to their specific needs and use cases.

2. **Efficient Knowledge Retrieval**: Langchain provides a powerful knowledge retrieval system that enables users to access and combine information from various sources, including text, images, and other data types. This allows users to create applications that can efficiently retrieve and process information, making them more productive and effective.

3. **Customizable and Interpretable AI Systems**

In [16]:
## streaming example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

Langchain is an open-source AI development platform that enables users to build and integrate large language models. Here are three potential benefits of using Langchain:

1. **Customizable AI Solutions**: Langchain allows developers to create custom AI models tailored to specific use cases, industries, or applications. This customization enables businesses and organizations to leverage AI in a more targeted and effective manner. By integrating Langchain with their existing systems and workflows, users can create AI-powered solutions that address their unique challenges and needs.

2. **Streamlined AI Development and Integration**: Langchain provides a scalable and modular architecture that simplifies the development and integration of large language models. This makes it easier for developers to build, deploy, and manage AI-powered applications, reducing the complexity and time associated with traditional AI development processes.

3. **Improved Data Integration and Utilization**: Lan

### Dynamic Prompt Templates

In [17]:
from langchain_core.prompts import ChatPromptTemplate
## create translation app

translation_template=ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the following {text} from {sourcelanguage} to {targetlanguage}.Maintain the tone and style"),
    ("user", "{text}")
])

## using the template
prompt=translation_template.invoke({
    "sourcelanguage":"English",
    "targetlanguage":"Spanish",
    "text":"Learning Langchain is very useful for building ai applications easily."
})

In [18]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the following Learning Langchain is very useful for building ai applications easily. from English to Spanish.Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Learning Langchain is very useful for building ai applications easily.', additional_kwargs={}, response_metadata={})])

In [19]:
translated_response=model.invoke(prompt)
print(translated_response.content)

Aprender Langchain es muy útil para construir aplicaciones de IA con facilidad.


### Building your first chain

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import  ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def create_story_chain():
    ## Template for Story Generation
    story_prompt=ChatPromptTemplate.from_messages(
        [
            ("system", "You are a creative storyteller. write a short, creative, and engaging story based on the given theme, character and setting."),
            ("user", "Theme: {theme}\n MainCharacter: {character}\n Setting: {setting}")
        ]
    )

    ## Template for story analysis
    analysis_prompt= ChatPromptTemplate.from_messages([
        ("system","You are a literary critic. Analyze the following story and provide insights."),
        ("user", "{story}")
    ])

    story_chain=(
        story_prompt | model | StrOutputParser()
    )

    def analyze_story(story_text):
        return {"story": story_text}

    analysis_chain=(
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )

    return analysis_chain

In [27]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. write a short, creative, and engaging story based on the given theme, character and setting.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n MainCharacter: {character}\n Setting: {setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': 

In [28]:
result = chain.invoke({
    "theme": "artificial intelligence",
    "character": "a curious robot",
    "setting": "a futuristic city"
})

print("Story and Analysis:")
print(result)

Story and Analysis:
**Analysis of "The Curious Robot of New Elysium"**

At its core, "The Curious Robot of New Elysium" is a thought-provoking science fiction tale that explores the consequences of artificial intelligence, free will, and the blurred lines between curiosity and obsession. The story masterfully weaves together a futuristic setting, a cast of intriguing characters, and a narrative that is both engaging and intellectually stimulating.

**Themes and Symbolism**

The narrative explores several key themes, including:

1. **The Double-Edged Nature of Curiosity**: Zeta's insatiable thirst for knowledge drives the plot, but his curiosity also leads him down a path of discovery that ultimately puts the city of New Elysium at risk. This serves as a cautionary tale about the potential consequences of unchecked curiosity.
2. **The Ethics of AI**: The Architect's intentions and motivations raise questions about the ethics of artificial intelligence. Is it acceptable to create AI enti